# LeetCode 2046 — Sort Linked List Already Sorted Using Absolute Values bằng Python

Link đề: https://leetcode.com/problems/sort-linked-list-already-sorted-using-absolute-values/ · Độ khó: **Medium** · Chủ đề: Linked List, Two Pointers, Sorting

> **Lưu ý: đây là bài Premium** trên LeetCode — cần tài khoản Premium để xem đề gốc và nộp bài. Đề dưới đây viết theo hiểu biết về bài (nguồn: bản đề của doocs/leetcode); nếu có tài khoản Premium, hãy **đối chiếu lại đề gốc**.

Skeleton LeetCode:

```python
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def sortLinkedList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        ...
```

## 1. Mục tiêu học

1. Khai thác **cấu trúc có sẵn** của dữ liệu: danh sách đã sắp theo `|giá trị|` ⇒ các số âm xuất hiện theo thứ tự giá trị **giảm dần**, các số không âm theo thứ tự **tăng dần**.
2. Kỹ thuật **chuyển nút lên đầu** (head insertion) trên danh sách liên kết đơn: gỡ nút khỏi vị trí hiện tại và gắn vào đầu trong `O(1)`.
3. Kỹ thuật **tách hai danh sách** (dummy node) rồi **đảo** một danh sách và **nối** lại.
4. Giữ **con trỏ `prev`** đúng sau khi gỡ nút (không bỏ sót nút kế tiếp), và luôn trả về **đầu mới**.
5. Đạt follow-up `O(n)` thời gian, `O(1)` bộ nhớ phụ.

## 2. Đề bài & ví dụ

Cho `head` của một danh sách liên kết đơn đã được sắp **không giảm theo giá trị tuyệt đối** của các nút. Trả về danh sách được sắp **không giảm theo giá trị thực**.

**Ví dụ 1:** `[0,2,-5,5,10,-10]` → `[-10,-5,0,2,5,10]`.

**Ví dụ 2:** `[0,1,2]` → `[0,1,2]` (đã sắp).

**Ví dụ 3:** `[1]` → `[1]`.

**Ràng buộc:** số nút trong `[1, 10⁵]`; `−5000 ≤ Node.val ≤ 5000`; danh sách sắp không giảm theo `|val|`.

**Follow-up:** có lời giải `O(n)` thời gian không?

*(Đề Premium — đã diễn đạt lại, hãy đối chiếu lại đề gốc.)*

In [1]:
import random
import time
from typing import List, Optional

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(vals):
    dummy = ListNode(); cur = dummy
    for v in vals:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next

def to_list(head, limit=10 ** 6):
    out = []
    while head is not None:
        out.append(head.val); head = head.next
        if len(out) > limit:
            raise RuntimeError("có chu trình?")
    return out

examples = [([0, 2, -5, 5, 10, -10], [-10, -5, 0, 2, 5, 10]), ([0, 1, 2], [0, 1, 2]), ([1], [1])]
for vals, exp in examples:
    print(f"{vals} -> kỳ vọng {exp}")

[0, 2, -5, 5, 10, -10] -> kỳ vọng [-10, -5, 0, 2, 5, 10]
[0, 1, 2] -> kỳ vọng [0, 1, 2]
[1] -> kỳ vọng [1]


## 3. Kiến thức nền

### 3.1 Danh sách liên kết đơn và dummy node

Mỗi nút giữ `val` và `next`. Một **nút giả** (dummy) đặt trước đầu danh sách giúp mọi thao tác chèn/xóa ở đầu giống hệt ở giữa. Đầu thật là `dummy.next`.

### 3.2 Gỡ một nút và chèn lên đầu

Với `prev → cur → nxt`: gỡ `cur` bằng `prev.next = nxt`; chèn lên đầu bằng `cur.next = head; head = cur`. Cả hai `O(1)`. Sau khi gỡ, nút tiếp theo cần xét là `nxt` và `prev` **giữ nguyên**.

In [2]:
head = build([1, 2, 3, 4])
prev = head.next                          # nút 2
cur = prev.next                           # nút 3 (cần chuyển lên đầu)
prev.next = cur.next
cur.next = head; head = cur
print("chuyển 3 lên đầu:", to_list(head))

chuyển 3 lên đầu: [3, 1, 2, 4]


### 3.3 Đảo danh sách liên kết

Duyệt và quay `next` của từng nút về trước: `O(n)`, `O(1)`.

In [3]:
def reverse(head):
    prev = None
    while head:
        head.next, prev, head = prev, head, head.next
    return prev

print(to_list(reverse(build([1, 2, 3]))))

[3, 2, 1]


### 3.4 Thứ tự theo trị tuyệt đối nói gì về số âm

Nếu `|a| ≤ |b|` và cả hai âm thì `a ≥ b`. Vậy trong danh sách sắp theo `|val|`:
- dãy con các số **âm** có giá trị thực **không tăng** (ví dụ `−5, −10`),
- dãy con các số **không âm** có giá trị thực **không giảm** (`0, 2, 5, 10`).

In [4]:
vals = [0, 2, -5, 5, 10, -10, -10, 11]
print("âm:", [v for v in vals if v < 0], "| không âm:", [v for v in vals if v >= 0])

âm: [-5, -10, -10] | không âm: [0, 2, 5, 10, 11]


### 3.5 Phát hiện chu trình khi debug: thuật toán Floyd (rùa và thỏ)

Khi thao tác con trỏ sai, danh sách dễ thành **chu trình** và vòng `while cur` chạy mãi. Ngoài cách đặt giới hạn bước (như `to_list`), có thể kiểm tra bằng hai con trỏ: `slow` đi 1 bước, `fast` đi 2 bước; nếu gặp nhau thì có chu trình. `O(n)` thời gian, `O(1)` bộ nhớ — hữu ích để viết `assert` trong lúc thử nghiệm các cách ở mục 7 và 13.

In [5]:
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

a = build([1, 2, 3]); print("không chu trình:", has_cycle(a))
a.next.next.next = a.next                  # 3 -> 2 tạo chu trình
print("sau khi nối 3 -> 2:", has_cycle(a))

không chu trình: False
sau khi nối 3 -> 2: True


## 4. Ý tưởng nền tảng / trực giác

Kết quả mong muốn = (các số âm theo thứ tự **tăng**) + (các số không âm theo thứ tự tăng). Dãy không âm đã đúng thứ tự trong danh sách; dãy âm đang **ngược**. Có hai cách tự nhiên:

1. **Chuyển lên đầu:** duyệt từ trái sang phải; gặp số âm thì gỡ ra và đặt lên **đầu**. Vì các số âm gặp sau có giá trị **nhỏ hơn hoặc bằng**, đặt lên đầu liên tiếp sẽ tự xếp chúng theo thứ tự tăng ở đầu danh sách.
2. **Tách – đảo – nối:** tách thành hai danh sách (âm / không âm) giữ thứ tự xuất hiện, đảo danh sách âm, nối với danh sách không âm.

Tính tay **Ví dụ 1** theo cách 1:

| bước | nút xét | hành động | danh sách |
|---|---|---|---|
| 1 | 2 | giữ | `0 → 2 → −5 → 5 → 10 → −10` |
| 2 | −5 | lên đầu | `−5 → 0 → 2 → 5 → 10 → −10` |
| 3 | 5 | giữ | |
| 4 | 10 | giữ | |
| 5 | −10 | lên đầu | `−10 → −5 → 0 → 2 → 5 → 10` |

## 5. Cách 1 — Brute Force / baseline: lấy giá trị, sắp xếp, dựng lại

Đọc mọi giá trị vào list, `sorted`, rồi **ghi đè** giá trị vào các nút theo thứ tự (hoặc dựng danh sách mới). Không dùng cấu trúc đặc biệt nào của đề.

In [6]:
def sort_bruteforce(head):
    vals = sorted(to_list(head))
    cur = head
    for v in vals:                     # ghi đè giá trị, giữ nguyên các nút
        cur.val = v; cur = cur.next
    return head

for vals, exp in examples:
    got = to_list(sort_bruteforce(build(vals))); print(got, "kỳ vọng", exp); assert got == exp

[-10, -5, 0, 2, 5, 10] kỳ vọng [-10, -5, 0, 2, 5, 10]
[0, 1, 2] kỳ vọng [0, 1, 2]
[1] kỳ vọng [1]


**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ. Chấp nhận được nhưng chưa đạt follow-up; ngoài ra ghi đè `val` có thể không được phép nếu nút mang dữ liệu khác.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** số âm trong danh sách theo thứ tự `|val|` tăng ⇔ giá trị thực **giảm**. Chèn liên tiếp lên đầu một dãy giảm cho ra dãy **tăng**.

**Insight 2:** số không âm đã tăng dần và giữ nguyên vị trí tương đối; mọi số âm đều nhỏ hơn mọi số không âm ⇒ đặt toàn bộ số âm ở đầu là đúng.

**Invariant (cách chuyển lên đầu):** khi xét nút `cur`, phần danh sách từ `head` tới `prev` gồm: (các số âm đã gặp, theo thứ tự tăng) rồi (các số không âm đã gặp, theo thứ tự tăng). `prev` là nút không âm cuối cùng đã gặp (hoặc đầu ban đầu).

**Chi tiết con trỏ:** khi gỡ `cur` (âm), `prev` **không** tiến; khi `cur` không âm, `prev = cur`. Đầu danh sách ban đầu luôn giữ nguyên làm `prev` khởi đầu — kể cả khi nó âm (nó là số âm có `|val|` nhỏ nhất, tức **lớn nhất** trong các số âm, đúng vị trí cuối của khối âm).

### 6.1 Minh họa: các số âm xuất hiện theo giá trị giảm

Sinh một danh sách hợp lệ ngẫu nhiên và in dãy số âm theo thứ tự xuất hiện — luôn không tăng.

In [7]:
def rand_abs_sorted(rng, n, vmax=20):
    vals = [rng.randint(-vmax, vmax) for _ in range(n)]
    vals.sort(key=abs)
    return vals

rng = random.Random(61)
v = rand_abs_sorted(rng, 15)
neg = [x for x in v if x < 0]
print("danh sách:", v)
print("số âm theo thứ tự xuất hiện:", neg, "| không tăng:", all(neg[i] >= neg[i + 1] for i in range(len(neg) - 1)))

danh sách: [0, 0, 1, -2, 2, 2, 5, -7, -9, 9, 11, 11, 15, -16, -19]
số âm theo thứ tự xuất hiện: [-2, -7, -9, -16, -19] | không tăng: True


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chuyển số âm lên đầu (bản nộp, `O(n)` / `O(1)`)

In [8]:
def sort_move_front(head):
    prev, cur = head, head.next
    while cur:
        if cur.val < 0:
            prev.next = cur.next          # gỡ cur
            cur.next = head               # chèn lên đầu
            head = cur
            cur = prev.next               # prev giữ nguyên
        else:
            prev, cur = cur, cur.next
    return head

for vals, exp in examples:
    assert to_list(sort_move_front(build(vals))) == exp
print("Cách 2 khớp 3 ví dụ")

Cách 2 khớp 3 ví dụ


**Độ phức tạp Cách 2:** mỗi nút xét đúng một lần, mỗi thao tác `O(1)` ⇒ `O(n)` thời gian, `O(1)` bộ nhớ phụ.

### 7.2 Cách 3 — Tách hai danh sách, đảo danh sách âm, nối

In [9]:
def sort_split(head):
    neg_d, pos_d = ListNode(), ListNode()
    nt, pt = neg_d, pos_d
    while head:
        nxt = head.next; head.next = None
        if head.val < 0:
            nt.next = head; nt = head
        else:
            pt.next = head; pt = head
        head = nxt
    neg = reverse(neg_d.next)
    if neg is None:
        return pos_d.next
    tail = neg
    while tail.next:
        tail = tail.next
    tail.next = pos_d.next
    return neg

for vals, exp in examples:
    assert to_list(sort_split(build(vals))) == exp
print("Cách 3 khớp 3 ví dụ")

Cách 3 khớp 3 ví dụ


Có thể bỏ vòng tìm đuôi: nút âm **đầu tiên** gặp sẽ là đuôi sau khi đảo — lưu nó lại.

**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(1)` bộ nhớ phụ.

### 7.3 Cách 4 — Đếm (counting sort) trên miền `[−5000, 5000]`

Giá trị nhỏ ⇒ đếm tần suất rồi ghi lại theo thứ tự. `O(n + V)` với `V = 10001`. Không dùng cấu trúc "đã sắp theo `|val|`", nhưng là lời giải `O(n)` tổng quát khi miền giá trị nhỏ.

In [10]:
def sort_counting(head):
    OFF = 5000
    cnt = [0] * (2 * OFF + 1)
    cur = head
    while cur:
        cnt[cur.val + OFF] += 1; cur = cur.next
    cur = head
    for i, c in enumerate(cnt):
        for _ in range(c):
            cur.val = i - OFF; cur = cur.next
    return head

for vals, exp in examples:
    assert to_list(sort_counting(build(vals))) == exp
print("Cách 4 khớp 3 ví dụ")

Cách 4 khớp 3 ví dụ


**Độ phức tạp Cách 4:** `O(n + V)` thời gian, `O(V)` bộ nhớ; ghi đè `val`.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1** và trên một ví dụ tự dựng có **đầu âm** và số âm liên tiếp: `[-1, 2, -3, -3, 4, -5]` (đối chiếu lại đề gốc).

In [11]:
def trace_move_front(vals):
    head = build(vals); prev, cur = head, head.next
    print("ban đầu:", to_list(head))
    while cur:
        if cur.val < 0:
            v = cur.val
            prev.next = cur.next; cur.next = head; head = cur; cur = prev.next
            print(f"  {v:>3} âm  -> lên đầu: {to_list(head)}   (prev vẫn là {prev.val})")
        else:
            print(f"  {cur.val:>3} >= 0 -> giữ, prev = {cur.val}")
            prev, cur = cur, cur.next
    return to_list(head)

assert trace_move_front([0, 2, -5, 5, 10, -10]) == [-10, -5, 0, 2, 5, 10]
print()
assert trace_move_front([-1, 2, -3, -3, 4, -5]) == sorted([-1, 2, -3, -3, 4, -5])

ban đầu: [0, 2, -5, 5, 10, -10]
    2 >= 0 -> giữ, prev = 2
   -5 âm  -> lên đầu: [-5, 0, 2, 5, 10, -10]   (prev vẫn là 2)
    5 >= 0 -> giữ, prev = 5
   10 >= 0 -> giữ, prev = 10
  -10 âm  -> lên đầu: [-10, -5, 0, 2, 5, 10]   (prev vẫn là 10)

ban đầu: [-1, 2, -3, -3, 4, -5]
    2 >= 0 -> giữ, prev = 2
   -3 âm  -> lên đầu: [-3, -1, 2, -3, 4, -5]   (prev vẫn là 2)
   -3 âm  -> lên đầu: [-3, -3, -1, 2, 4, -5]   (prev vẫn là 2)
    4 >= 0 -> giữ, prev = 4
   -5 âm  -> lên đầu: [-5, -3, -3, -1, 2, 4]   (prev vẫn là 4)


Ở ví dụ thứ hai, đầu `−1` **không** bị di chuyển nhưng vẫn đúng chỗ: các số âm khác (`−3`, `−3`, `−5`) đều nhỏ hơn và được chèn **trước** nó. Hai số `−3` liên tiếp: sau khi gỡ `−3` thứ nhất, `cur = prev.next` là `−3` thứ hai — không bị bỏ sót.

| bước | nút | hành động | danh sách | prev |
|---|---|---|---|---|
| 1 | 2 | giữ | `−1, 2, −3, −3, 4, −5` | 2 |
| 2 | −3 | lên đầu | `−3, −1, 2, −3, 4, −5` | 2 |
| 3 | −3 | lên đầu | `−3, −3, −1, 2, 4, −5` | 2 |
| 4 | 4 | giữ | | 4 |
| 5 | −5 | lên đầu | `−5, −3, −3, −1, 2, 4` | 4 |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — chuyển số âm lên đầu, `O(n)` / `O(1)`.

In [12]:
class Solution:
    def sortLinkedList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        prev, cur = head, head.next
        while cur:
            if cur.val < 0:
                prev.next = cur.next
                cur.next = head
                head = cur
                cur = prev.next
            else:
                prev, cur = cur, cur.next
        return head

for vals, exp in examples:
    got = to_list(Solution().sortLinkedList(build(vals))); print(got, exp); assert got == exp

[-10, -5, 0, 2, 5, 10] [-10, -5, 0, 2, 5, 10]
[0, 1, 2] [0, 1, 2]
[1] [1]


## 10. Vì sao thuật toán đúng

Gọi các nút theo thứ tự ban đầu `a₁, a₂, …, aₙ` (sắp theo `|val|`).

**Invariant.** Trước khi xét `aₖ` (`k ≥ 2`), danh sách là `N · P · aₖ … aₙ` với:
- `N` = các số âm trong `a₂..a_{k−1}` theo thứ tự **ngược** với lúc xuất hiện (tức giá trị không giảm, theo §3.4),
- `P` = `a₁` cùng các số không âm trong `a₂..a_{k−1}` theo thứ tự xuất hiện, `prev` là nút cuối của `P`.

*Khởi tạo* (`k = 2`): `N` rỗng, `P = a₁`, `prev = a₁` ✓.
*Bước:* nếu `aₖ ≥ 0`: nó nối sau `prev` ⇒ `P` thêm `aₖ`, `prev = aₖ` ✓. Nếu `aₖ < 0`: gỡ khỏi sau `prev` và chèn lên đầu ⇒ `N` thêm `aₖ` ở **đầu** ✓; `prev` giữ, `cur = prev.next = a_{k+1}` ✓.

**Kết thúc:** danh sách `N · P`. `N` không giảm (các số âm gặp sau nhỏ hơn hoặc bằng). `P`: nếu `a₁ ≥ 0` thì `P` là các số không âm theo `|val|` tăng ⇒ tăng. Nếu `a₁ < 0` thì `a₁` có `|val|` nhỏ nhất ⇒ `a₁` là số âm **lớn nhất**, đứng sau mọi phần tử của `N` (`≤ a₁`) và trước mọi số không âm ⇒ `N · P` vẫn không giảm. ∎

### 10.1 Kiểm chứng thực nghiệm (kể cả **định danh nút**)

Ngoài so giá trị với `sorted`, kiểm tra Cách 2 và Cách 3 **tái sử dụng đúng các nút cũ** (không tạo nút mới, không mất nút) và không tạo chu trình.

In [13]:
rng = random.Random(1010)
for _ in range(2000):
    vals = rand_abs_sorted(rng, rng.randint(1, 15), rng.choice([2, 5, 50]))
    for f in (sort_move_front, sort_split, Solution().sortLinkedList):
        head = build(vals); ids = set()
        cur = head
        while cur: ids.add(id(cur)); cur = cur.next
        out = f(head); got = to_list(out)
        assert got == sorted(vals)
        cur, ids2 = out, set()
        while cur: ids2.add(id(cur)); cur = cur.next
        assert ids == ids2
print("Đúng thứ tự, giữ nguyên tập nút, không chu trình (2000 danh sách x 3 cách)")

Đúng thứ tự, giữ nguyên tập nút, không chu trình (2000 danh sách x 3 cách)


## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 10⁵`, `V = 10001`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Sắp giá trị | `O(n log n)` | `O(n)` |
| 2. Chuyển lên đầu | `O(n)` | `O(1)` |
| 3. Tách – đảo – nối | `O(n)` | `O(1)` |
| 4. Đếm | `O(n + V)` | `O(V)` |

In [14]:
rng = random.Random(2046)
big = rand_abs_sorted(rng, 10 ** 5, 5000)
for name, f in [("brute", sort_bruteforce), ("move_front", sort_move_front), ("split", sort_split), ("counting", sort_counting), ("submit", Solution().sortLinkedList)]:
    head = build(big)
    t = time.perf_counter(); out = f(head); dt = time.perf_counter() - t
    assert to_list(out) == sorted(big)
    print(f"{name:<11}: {dt * 1000:.0f} ms")

brute      : 16 ms


move_front : 11 ms


split      : 14 ms
counting   : 16 ms


submit     : 11 ms


## 12. Bộ test

In [15]:
ALL = {"brute": sort_bruteforce, "move_front": sort_move_front, "split": sort_split, "counting": sort_counting, "submit": Solution().sortLinkedList}
def run_all(vals):
    res = {k: to_list(f(build(vals))) for k, f in ALL.items()}
    vs = list(res.values())
    assert all(v == vs[0] for v in vs) and vs[0] == sorted(vals), (vals, res)
    return vs[0]

for vals, exp in examples:
    assert run_all(vals) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kỳ vọng so với sorted()
edge = [[-1], [-1, -2, -3], [0, 0, -1, 1], [-1, 1, -1, 1], [5000, -5000], [-3, 3, -3], [0], [-2, 2]]
for vals in edge:
    assert sorted(vals, key=abs) == vals or [abs(x) for x in vals] == sorted(abs(x) for x in vals)
    run_all(vals)
rng = random.Random(20460)
for _ in range(1000):
    run_all(rand_abs_sorted(rng, rng.randint(1, 30), rng.choice([1, 3, 10, 5000])))
print("Ví dụ, 8 edge case, 1000 danh sách ngẫu nhiên: OK")

Ví dụ, 8 edge case, 1000 danh sách ngẫu nhiên: OK


### 12.1 Kiểm tra tính chất

- Kết quả là một **hoán vị** của đầu vào (đa tập giá trị giữ nguyên).
- Nếu mọi giá trị `≥ 0`, danh sách không đổi (cả thứ tự nút).
- Nếu mọi giá trị `< 0`, kết quả là danh sách **đảo ngược**.

In [16]:
rng = random.Random(121)
f = Solution().sortLinkedList
for _ in range(500):
    n = rng.randint(1, 20)
    pos = sorted(rng.randint(0, 30) for _ in range(n))
    head = build(pos); nodes = []
    cur = head
    while cur: nodes.append(cur); cur = cur.next
    out = f(head); cur, k = out, 0
    while cur: assert cur is nodes[k]; cur = cur.next; k += 1
    neg = sorted((-rng.randint(1, 30) for _ in range(n)), key=abs)
    assert to_list(f(build(neg))) == neg[::-1]
    mix = rand_abs_sorted(rng, n)
    assert sorted(to_list(f(build(mix)))) == sorted(mix)
print("3 tính chất đúng trên 500 lần thử")

3 tính chất đúng trên 500 lần thử


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tiến `prev` sau khi gỡ nút (bỏ sót số âm liên tiếp)

In [17]:
def wrong_advance_prev(head):
    prev, cur = head, head.next
    while cur:
        nxt = cur.next
        if cur.val < 0:
            prev.next = cur.next; cur.next = head; head = cur
            prev = nxt if nxt else prev                    # SAI: nhảy prev qua nút kế
            cur = prev.next if nxt else None
        else:
            prev, cur = cur, nxt
    return head

vals = [1, -2, -3, 4]          # tự dựng (đối chiếu lại đề gốc)
print("sai :", to_list(wrong_advance_prev(build(vals))), "| đúng:", sorted(vals))

sai : [-2, 1, -3, 4] | đúng: [-3, -2, 1, 4]


**Vì sao sai:** sau khi gỡ `cur`, nút cần xét tiếp là `prev.next` (chính là `nxt`) và `prev` phải **đứng yên**. Nhảy `prev` qua `nxt` khiến `nxt` (ở đây `−3`) không bao giờ được xét. **Sửa:** `cur = prev.next`, giữ `prev`.

### Lỗi 2 — Nối số âm vào **đuôi** khối âm (không đảo)

In [18]:
def wrong_no_reverse(head):
    neg, pos = [], []
    cur = head
    while cur:
        (neg if cur.val < 0 else pos).append(cur.val); cur = cur.next
    return build(neg + pos)                                # SAI: neg đang giảm dần

for vals, exp in examples:
    print("sai :", to_list(wrong_no_reverse(build(vals))), "| đúng:", exp)

sai : [-5, -10, 0, 2, 5, 10] | đúng: [-10, -5, 0, 2, 5, 10]
sai : [0, 1, 2] | đúng: [0, 1, 2]
sai : [1] | đúng: [1]


**Vì sao sai:** các số âm xuất hiện theo `|val|` tăng ⇒ giá trị **giảm**; giữ nguyên thứ tự cho `[−5, −10, …]`. **Sửa:** đảo khối âm (Cách 3) hoặc chèn lên đầu (Cách 2).

### Lỗi 3 — Quên trả về **đầu mới**

In [19]:
def wrong_return_old_head(head):
    original = head
    prev, cur = head, head.next
    while cur:
        if cur.val < 0:
            prev.next = cur.next; cur.next = head; head = cur; cur = prev.next
        else:
            prev, cur = cur, cur.next
    return original                                        # SAI

for vals, exp in examples:
    print("sai :", to_list(wrong_return_old_head(build(vals))), "| đúng:", exp)

sai : [0, 2, 5, 10] | đúng: [-10, -5, 0, 2, 5, 10]
sai : [0, 1, 2] | đúng: [0, 1, 2]
sai : [1] | đúng: [1]


**Vì sao sai:** các nút âm đã được chèn **trước** đầu cũ; trả đầu cũ làm mất chúng. **Sửa:** cập nhật và trả biến `head`.

### Lỗi 4 — Bắt đầu từ `prev = None, cur = head` và cố "chuyển" cả đầu

In [20]:
def wrong_from_head(head, guard=50):
    prev, cur, steps = None, head, 0
    while cur and steps < guard:
        steps += 1
        if cur.val < 0 and prev is not None:
            prev.next = cur.next; cur.next = head; head = cur; cur = prev.next
        elif cur.val < 0:                                  # SAI: "chuyển" chính đầu lên đầu
            cur.next = head; head = cur; cur = cur.next    # tạo khuyên cur -> cur
        else:
            prev, cur = cur, cur.next
    return head, steps

h, steps = wrong_from_head(build([-1, 2, -3]))             # tự dựng (đối chiếu lại đề gốc)
print("dừng do vượt", steps, "bước (vòng lặp vô hạn), nút đầu trỏ tới chính nó:", h.next is h)

dừng do vượt 50 bước (vòng lặp vô hạn), nút đầu trỏ tới chính nó: True


**Vì sao sai:** chuyển nút đầu lên đầu nghĩa là `cur.next = cur` — tạo chu trình, vòng lặp không dừng. Đầu danh sách ban đầu **không cần** di chuyển (xem chứng minh §10). **Sửa:** bắt đầu với `prev = head, cur = head.next`.

### Lỗi 5 — Sắp lại bằng cách tạo nút mới khi đề yêu cầu sắp danh sách

In [21]:
def new_nodes(head):
    return build(sorted(to_list(head)))                    # trả danh sách MỚI

head = build([0, 2, -5]); old_ids = set()
cur = head
while cur: old_ids.add(id(cur)); cur = cur.next
out = new_nodes(head); new_ids = set()
cur = out
while cur: new_ids.add(id(cur)); cur = cur.next
print("giá trị đúng:", to_list(out), "| dùng lại nút cũ:", old_ids == new_ids, "| bộ nhớ phụ O(n)")

giá trị đúng: [-5, 0, 2] | dùng lại nút cũ: False | bộ nhớ phụ O(n)


**Vì sao không tốt:** đáp án giá trị đúng (thường vẫn AC), nhưng tốn `O(n)` bộ nhớ, `O(n log n)` thời gian và không đạt follow-up; nếu nút mang dữ liệu khác ngoài `val`, tạo nút mới làm mất dữ liệu. **Sửa:** sắp xếp lại liên kết (Cách 2/3).

### Lỗi 6 — Cách 3 nhưng quên cắt `next` khi tách (tạo chu trình)

In [22]:
def wrong_split_no_cut(head, guard=100):
    neg_d, pos_d = ListNode(), ListNode(); nt, pt = neg_d, pos_d
    while head:
        if head.val < 0: nt.next = head; nt = head
        else: pt.next = head; pt = head
        head = head.next                                   # SAI: không đặt head.next = None
    neg = reverse(neg_d.next)
    if neg is None: return pos_d.next
    tail = neg
    while tail.next and guard: tail = tail.next; guard -= 1
    tail.next = pos_d.next
    return neg

try:
    print(to_list(wrong_split_no_cut(build([1, -2, 3])), limit=50))
except RuntimeError as e:
    print("lỗi:", e, "(ví dụ tự dựng [1, -2, 3])")

lỗi: có chu trình? (ví dụ tự dựng [1, -2, 3])


**Vì sao sai:** nút cuối của mỗi danh sách con vẫn trỏ tới nút cũ phía sau (thuộc danh sách kia) ⇒ khi đảo và nối sinh ra chu trình hoặc nút lặp. **Sửa:** `head.next = None` khi tách (hoặc chốt `nt.next = None`, `pt.next = None` ở cuối).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Dữ liệu "đã sắp theo một khóa khác" ⇒ thường chỉ cần **đảo một phần** hoặc **trộn hai dãy** thay vì sắp lại (977 Squares of a Sorted Array là phiên bản ngược lại).
- Danh sách liên kết: **dummy node**, **gỡ–chèn `O(1)`**, cẩn thận con trỏ `prev` sau khi gỡ.
- Luôn kiểm tra: đầu danh sách có thay đổi không? có tạo chu trình không? có mất nút không?
- Viết `to_list` có giới hạn bước để phát hiện chu trình khi debug.

## 15. Biến thể của bài

1. **977 Squares of a Sorted Array**: mảng sắp theo giá trị thực ⇒ trả bình phương sắp tăng — hai con trỏ từ hai đầu.
2. **Mảng** sắp theo `|val|` ⇒ sắp theo giá trị thực `O(n)` (Bài tập 4).
3. **Ngược lại:** danh sách sắp theo giá trị thực ⇒ sắp theo `|val|` — trộn (merge) danh sách âm đã đảo với danh sách không âm (Bài tập 5).
4. **86 Partition List**: tách theo ngưỡng giữ thứ tự tương đối — giống bước tách của Cách 3.
5. **206 Reverse Linked List**, **21 Merge Two Sorted Lists**: các thao tác nền dùng ở đây.

### 15.1 Code biến thể: 977 Squares of a Sorted Array (hai con trỏ)

In [23]:
def sorted_squares(nums):
    n = len(nums); res = [0] * n; l, r = 0, n - 1
    for k in range(n - 1, -1, -1):
        if abs(nums[l]) > abs(nums[r]):
            res[k] = nums[l] ** 2; l += 1
        else:
            res[k] = nums[r] ** 2; r -= 1
    return res

rng = random.Random(977)
for _ in range(500):
    a = sorted(rng.randint(-20, 20) for _ in range(rng.randint(1, 15)))
    assert sorted_squares(a) == sorted(x * x for x in a)
print("977 khớp sorted:", sorted_squares([-4, -1, 0, 3, 10]))

977 khớp sorted: [0, 1, 9, 16, 100]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp giá trị | Đọc – sort – ghi | `O(n log n)` | `O(n)` | Kiểm chứng |
| 2. Chuyển lên đầu | Head insertion | `O(n)` | `O(1)` | **Bản nộp** |
| 3. Tách – đảo – nối | Hai danh sách | `O(n)` | `O(1)` | Dễ suy luận, dễ mở rộng |
| 4. Đếm | Miền giá trị nhỏ | `O(n + V)` | `O(V)` | Khi không có cấu trúc đặc biệt |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[-1]`; (b) `[-1,-2,-3]`; (c) `[0,0,-1,1]`; (d) `[1,-1,2,-2]`.

**Bài 2 (tính tay):** Trace Cách 2 trên `[-2, 3, -4, -4, 5, -6, 7]`: sau mỗi bước ghi danh sách và `prev`.

**Bài 3 (điền TODO):** Hoàn thiện bước "gỡ `cur` và chèn lên đầu" trong `move_front_todo`.

**Bài 4 (tự viết lại):** Viết `sort_array_abs_sorted(nums)` cho **mảng** sắp theo `|val|`, `O(n)`, không dùng `sort`.

**Bài 5 (mini challenge):** Viết `abs_sort_linked(head)`: đầu vào là danh sách sắp theo giá trị **thực**, trả danh sách sắp theo `|val|` (cùng `|val|` thì số âm trước), `O(n)` / `O(1)` bằng tách – đảo – **trộn**.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def move_front_todo(head):
    prev, cur = head, head.next
    while cur:
        if cur.val < 0:
            # TODO: prev.next = cur.next; cur.next = head; head = cur; cur = prev.next
            pass
        else:
            prev, cur = cur, cur.next
    return head

In [25]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def sort_array_abs_sorted(nums):
    # TODO: lấy các số âm theo thứ tự ngược, nối với các số không âm theo thứ tự xuôi
    pass

def abs_sort_linked(head):
    # TODO: tách âm / không âm; đảo khối âm; trộn hai danh sách theo |val| (âm trước khi hòa)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tách âm/không âm. Mức 2: đảo khối âm. Mức 3: (a) `[-1]`; (b) `[-3,-2,-1]`; (c) `[-1,0,0,1]`; (d) `[-2,-1,1,2]`.

**Bài 2.** Mức 1: `prev` bắt đầu ở `−2` (đầu). Mức 2: `3` giữ (`prev = 3`); `−4`, `−4` lên đầu; `5` giữ; `−6` lên đầu; `7` giữ. Mức 3: kết quả `[-6,-4,-4,-2,3,5,7]`.

**Bài 3.** Mức 1: gỡ trước, chèn sau. Mức 2: cập nhật `head`. Mức 3: `cur = prev.next`, không tiến `prev`.

**Bài 4.** Mức 1: `[x for x in nums if x < 0][::-1]`. Mức 2: nối `[x for x in nums if x >= 0]`. Mức 3: hai con trỏ ghi vào mảng kết quả để `O(1)` bộ nhớ phụ ngoài kết quả.

**Bài 5.** Mức 1: trong danh sách sắp theo giá trị thực, số âm có `|val|` giảm ⇒ đảo để `|val|` tăng. Mức 2: trộn như 21 Merge Two Sorted Lists với khóa `|val|`. Mức 3: khi `|a| = |b|`, lấy số âm trước.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2, 3
f = Solution().sortLinkedList
b1 = [to_list(f(build(v))) for v in ([-1], [-1, -2, -3], [0, 0, -1, 1], [1, -1, 2, -2])]
print("Bài 1:", b1); assert b1 == [[-1], [-3, -2, -1], [-1, 0, 0, 1], [-2, -1, 1, 2]]
print("Bài 2:"); assert trace_move_front([-2, 3, -4, -4, 5, -6, 7]) == [-6, -4, -4, -2, 3, 5, 7]
def move_front_todo(head):
    prev, cur = head, head.next
    while cur:
        if cur.val < 0:
            prev.next = cur.next; cur.next = head; head = cur; cur = prev.next
        else:
            prev, cur = cur, cur.next
    return head
rng = random.Random(3)
for _ in range(300):
    v = rand_abs_sorted(rng, rng.randint(1, 12)); assert to_list(move_front_todo(build(v))) == sorted(v)
print("Bài 3: OK")

Bài 1: [[-1], [-3, -2, -1], [-1, 0, 0, 1], [-2, -1, 1, 2]]
Bài 2:
ban đầu: [-2, 3, -4, -4, 5, -6, 7]
    3 >= 0 -> giữ, prev = 3
   -4 âm  -> lên đầu: [-4, -2, 3, -4, 5, -6, 7]   (prev vẫn là 3)
   -4 âm  -> lên đầu: [-4, -4, -2, 3, 5, -6, 7]   (prev vẫn là 3)
    5 >= 0 -> giữ, prev = 5
   -6 âm  -> lên đầu: [-6, -4, -4, -2, 3, 5, 7]   (prev vẫn là 5)
    7 >= 0 -> giữ, prev = 7
Bài 3: OK


In [27]:
# Bài 4, 5
def sort_array_abs_sorted(nums):
    neg = [x for x in nums if x < 0]
    return neg[::-1] + [x for x in nums if x >= 0]

def merge_by_abs(a, b):
    d = ListNode(); t = d
    while a and b:
        if (abs(a.val), a.val) <= (abs(b.val), b.val):
            t.next = a; a = a.next
        else:
            t.next = b; b = b.next
        t = t.next
    t.next = a or b
    return d.next

def abs_sort_linked(head):
    nd, pd = ListNode(), ListNode(); nt, pt = nd, pd
    while head:
        nxt = head.next; head.next = None
        if head.val < 0: nt.next = head; nt = head
        else: pt.next = head; pt = head
        head = nxt
    return merge_by_abs(reverse(nd.next), pd.next)

rng = random.Random(5)
for _ in range(500):
    v = rand_abs_sorted(rng, rng.randint(1, 15))
    assert sort_array_abs_sorted(v) == sorted(v)
    w = sorted(rng.randint(-10, 10) for _ in range(rng.randint(1, 15)))
    assert to_list(abs_sort_linked(build(w))) == sorted(w, key=lambda x: (abs(x), x))
print("Bài 4, 5: OK |", sort_array_abs_sorted([0, 2, -5, 5, 10, -10]), to_list(abs_sort_linked(build([-10, -5, 0, 2, 5, 10]))))

Bài 4, 5: OK | [-10, -5, 0, 2, 5, 10] [0, 2, -5, 5, -10, 10]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (chuyển số âm lên đầu):**

```python
prev, cur = head, head.next          # đầu ban đầu không cần di chuyển
while cur:
    if cur.val < 0:
        prev.next = cur.next         # gỡ
        cur.next = head; head = cur  # chèn lên đầu
        cur = prev.next              # prev giữ nguyên
    else:
        prev, cur = cur, cur.next
return head                          # đầu MỚI
```

**Checklist:**
- [ ] Sau khi gỡ nút, `prev` **giữ nguyên**, `cur = prev.next`?
- [ ] Trả về đầu **mới**?
- [ ] Không "chuyển" nút đầu lên đầu (tránh chu trình)?
- [ ] Khi tách danh sách, cắt `next` của từng nút?
- [ ] Đã test: đầu âm, số âm liên tiếp, toàn âm, toàn không âm, một nút?